# U-Net strain prediction

Standalone run on the supplied **SWG 256×256 geometry, 12 µm × 12 µm**. Edit the **Parameters** cell, then select **Kernel → Restart Kernel and Run All Cells**. Paths are relative to the repository root. Results are saved under `outputs/`.

**Required:** a matching trained checkpoint. Train it with `train.py` or put your manuscript weights at the `weights_file` path. The checkpoint is not bundled. Material, loading and training domain are encoded in the weights.


In [ ]:
%matplotlib inline
from pathlib import Path
import os
import sys

root_candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in root_candidates if (p / "plate_models.py").is_file()
             and (p / "data/swg_256.npz").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or its notebooks directory")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository:", ROOT)

from pathlib import Path
import numpy as np
import torch
from surrogate_model import UNet


## Parameters

Change the input file/key, physical domain size, physical parameters and output path here. Support maps use 1 for supports and 0 for suspended film. `width_um` sets **both** length and width of the square domain; independent unequal x/y extents are not exposed by these solvers. Changing the domain size scales every feature; change the input pixels to change individual teeth or gaps.
Coordinates are in µm; output strain is in percent. Grid dimensions must be divisible by 32. Use weights matched to the material, loading and domain.


In [ ]:
structure_file = "data/swg_256.npz"
structure_key = "array"
width_um = 12.0
weights_file = "outputs/training/best.pth"
output_file = "outputs/swg_surrogate.npz"


## Calculate and save

Input shape is `(batch,3,H,W)` in `[support,x_um,y_um]` order; spatial dimensions must be divisible by 32. The output is in percent strain. The manuscript checkpoint predicts approximately 0.190790% for SWG. Material parameters and load are encoded in the trained weights, not supplied to `UNet.forward`. To change material, loading or the training domain, generate the corresponding dataset and retrain. A toy checkpoint does not reproduce the manuscript SWG prediction.

In [ ]:
with np.load(structure_file) as data:
    support = data[structure_key]
ny, nx = support.shape
grid_y, grid_x = torch.meshgrid(torch.linspace(0, width_um, ny),
                              torch.linspace(0, width_um, nx), indexing="ij")
inputs = torch.stack([
    torch.tensor(support, dtype=torch.float32), grid_x, grid_y
])[None]
model = UNet().eval()
model.load_state_dict(torch.load(weights_file, map_location="cpu", weights_only=True))
with torch.no_grad():
    strain_percent = model(inputs)[0, 0].numpy()
Path(output_file).parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(output_file, support_map=support, strain_percent=strain_percent,
                    strain_fraction=strain_percent / 100, width_m=width_um * 1e-6)
print("Predicted strain peak (%):", strain_percent.max())


## Inspect the result

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
im = ax.imshow(strain_percent, origin="lower", extent=[0, width_um, 0, width_um])
ax.set(title="Predicted strain (%)", xlabel="x (um)", ylabel="y (um)")
fig.colorbar(im, ax=ax)
plt.show()
